# Subsequent Search Misses and Attentional Blinks
temporal effects on LWS probability, with reference to (1) the start of the trial; (2) the most recent target detection - while taking into account the "type" of the target

In [ ]:
import os

import numpy as np
import pandas as pd

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

import config as cnfg
from analysis.helpers.read_data import load_analysis_data
from analysis.helpers.visit_classification import (
    VisitType, classify_visit, enrich_funnel_with_history,
)

pio.renderers.default = "notebook"      # "notebook" or "browser"

## Prepare Data

In [ ]:
data, funnel_results = load_analysis_data(funnel_type="lws", event_type="visit")

# drop irrelevant columns
funnel_results = funnel_results.drop(columns=[
    "upto_gaze_coverage", "upto_fixation_rate", "upto_no_bad_action", "upto_no_miss_with_false_alarm",
    "upto_before_identification", "upto_after_identification", "upto_not_close_to_trial_end", "upto_not_before_exemplar_visit"
], errors="ignore")

In [ ]:
metadata = data.metadata
fixations = data.fixations
idents = data.identifications
targets = data.targets

hits = (
    idents
    .loc[idents["identification_category"] == "hit"]
    .drop(columns=[
        "identification_category", 'left_x', 'left_y', 'left_pupil', 'right_x', 'right_y', 'right_pupil'
    ])
    .merge(
        targets[["subject", "trial", "target", "category"]], on=["subject", "trial", "target"], how="left"
    )
    .rename(columns={"category": "target_category"})
    .sort_values(["subject", "trial", "time"])
    .reset_index(drop=True)
)
del idents

funnel_results = funnel_results.merge(
    metadata[["subject", "trial", "num_targets"]], on=["subject", "trial"], how="left"
)
del metadata

### Classify Visits
We can consider LWS sub-types based on the number and type of targets found so far in the trial, as well as non-LWS visits - e.g., the fixation/event co-occuring with target identification or returning to a previously identified target.

In [ ]:
funnel_results = enrich_funnel_with_history(funnel_results, hits, fixations)
funnel_results['visit_type'] = funnel_results.apply(lambda row: classify_visit(row, hits), axis=1)

In [8]:
funnel_results["visit_type"].value_counts(dropna=False)

visit_type
VisitType.IDENTIFICATION_VISIT        1007
VisitType.TARGET_RETURN                529
VisitType.LWS_BEFORE_ANY_HIT           342
VisitType.LWS_AFTER_HIT_DIFF           225
VisitType.OTHER                        191
VisitType.LWS_AFTER_2HIT_MIXED          25
VisitType.LWS_AFTER_HIT_SAME            16
VisitType.LWS_AFTER_2HIT_BOTH_DIFF       6
Name: count, dtype: int64